# Phase 7 V3 — Lightweight Improvements (No Retraining)
## Squeeze More F1 from Existing Models

**Current:** 3rd place, LB = 0.92170 (weighted vote DeBERTa 5-fold + SVC + LR)
**Target:** 0.95081 (1st place), gap = 0.029

### Strategy: No DeBERTa retraining, all improvements are fast (<5 min)

| # | Technique | Time | Why |
|---|-----------|------|-----|
| 1 | **XGBoost/LightGBM 5-fold** on TF-IDF+features | ~2 min | Add diversity — tree models see data differently than linear models |
| 2 | **4-model stacking** with XGB meta-learner | ~1 min | XGB meta-learner is stronger than LR for combining heterogeneous models |
| 3 | **Per-class threshold optimization** | ~1 min | Boost DeepSeek/Grok (weakest classes) recall |
| 4 | **Pseudo-label confident test samples** → retrain SVC/LR | ~2 min | Free extra training data from high-confidence predictions |
| 5 | **Fine-grained weight search** | ~1 min | Grid search with 0.01 step instead of 0.05 |

In [1]:
import os, warnings, gc, time, json, re, string
os.chdir('/Users/aliivaezii/Documents/MALTO')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from collections import Counter
from scipy import sparse
from scipy.sparse import hstack, csr_matrix

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}
np.random.seed(SEED)

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
y_all = train_df['LABEL'].values

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Labels: {dict(zip(*np.unique(y_all, return_counts=True)))}')

Train: (2400, 2), Test: (600, 2)
Labels: {np.int64(0): np.int64(1520), np.int64(1): np.int64(80), np.int64(2): np.int64(160), np.int64(3): np.int64(80), np.int64(4): np.int64(240), np.int64(5): np.int64(320)}


## 1. Load All Existing Artifacts

We already have:
- DeBERTa 5-fold OOF probs (2400 samples)
- SVC 5-fold OOF probs (2400 samples)
- LR 5-fold OOF probs (2400 samples)
- TF-IDF vectorizers + scaler + handcrafted features

In [2]:
# Load DeBERTa 5-fold artifacts
deberta_oof_probs = np.load('models/deberta_kfold_oof_probs.npy')
deberta_test_probs = np.load('models/deberta_kfold_test_probs.npy')

# Load SVC/LR OOF artifacts
svc_oof_probs = np.load('models/ensemble_svc_oof_probs.npy')
lr_oof_probs = np.load('models/ensemble_lr_oof_probs.npy')
svc_test_probs = np.load('models/ensemble_svc_test_probs_cv.npy')
lr_test_probs = np.load('models/ensemble_lr_test_probs_cv.npy')

# Sanity check
print('Loaded OOF probabilities:')
for name, probs in [('DeBERTa', deberta_oof_probs), ('SVC', svc_oof_probs), ('LR', lr_oof_probs)]:
    f1 = f1_score(y_all, probs.argmax(1), average='macro')
    print(f'  {name:10s}: {probs.shape}, OOF F1={f1:.4f}')

# Current best: weighted vote
best_blend = 0.65 * deberta_oof_probs + 0.25 * svc_oof_probs + 0.10 * lr_oof_probs
best_f1 = f1_score(y_all, best_blend.argmax(1), average='macro')
print(f'\nCurrent best weighted vote OOF F1: {best_f1:.4f}')

Loaded OOF probabilities:
  DeBERTa   : (2400, 6), OOF F1=0.9428
  SVC       : (2400, 6), OOF F1=0.8979
  LR        : (2400, 6), OOF F1=0.9026

Current best weighted vote OOF F1: 0.9500


## 2. Train XGBoost & LightGBM on TF-IDF (5-fold OOF)

Tree-based models provide **different error patterns** from linear SVC/LR. Adding them to the ensemble creates diversity, which is the key to better ensembles.

In [3]:
# Rebuild TF-IDF features (fast, <10 seconds)
from src.features import extract_features

print('Extracting handcrafted features...')
train_feats = extract_features(train_df['TEXT'].values)
test_feats = extract_features(test_df['TEXT'].values)

print('Building TF-IDF features...')
tfidf_word = joblib.load('models/tfidf_word.joblib')
tfidf_char = joblib.load('models/tfidf_char.joblib')
scaler = joblib.load('models/feature_scaler.joblib')

X_word_train = tfidf_word.transform(train_df['TEXT'])
X_char_train = tfidf_char.transform(train_df['TEXT'])
X_feat_train = csr_matrix(scaler.transform(train_feats))
X_train_full = hstack([X_word_train, X_char_train, X_feat_train])

X_word_test = tfidf_word.transform(test_df['TEXT'])
X_char_test = tfidf_char.transform(test_df['TEXT'])
X_feat_test = csr_matrix(scaler.transform(test_feats))
X_test_full = hstack([X_word_test, X_char_test, X_feat_test])

print(f'Train features: {X_train_full.shape}')
print(f'Test features: {X_test_full.shape}')

Extracting handcrafted features...


Building TF-IDF features...
Train features: (2400, 100046)
Test features: (600, 100046)


In [5]:
# 5-fold XGBoost OOF
print('Training XGBoost 5-fold...')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

xgb_oof_probs = np.zeros((len(y_all), NUM_LABELS))
xgb_test_probs = np.zeros((len(test_df), NUM_LABELS))

# Compute sample weights for class imbalance
class_counts = np.bincount(y_all)
class_weight_map = {c: len(y_all) / (NUM_LABELS * class_counts[c]) for c in range(NUM_LABELS)}
sample_weights_all = np.array([class_weight_map[y] for y in y_all])

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_train_full, y_all)):
    X_tr = X_train_full[tr_idx]
    X_vl = X_train_full[vl_idx]
    y_tr = y_all[tr_idx]
    y_vl = y_all[vl_idx]
    w_tr = sample_weights_all[tr_idx]

    xgb_model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.3,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=SEED,
        eval_metric='mlogloss',
        tree_method='hist',
        early_stopping_rounds=50,
        n_jobs=-1,
    )
    xgb_model.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_vl, y_vl)], verbose=False)

    xgb_oof_probs[vl_idx] = xgb_model.predict_proba(X_vl)
    xgb_test_probs += xgb_model.predict_proba(X_test_full) / 5

    fold_f1 = f1_score(y_vl, xgb_oof_probs[vl_idx].argmax(1), average='macro')
    print(f'  Fold {fold+1}: F1={fold_f1:.4f}')

xgb_oof_f1 = f1_score(y_all, xgb_oof_probs.argmax(1), average='macro')
print(f'\nXGBoost OOF F1: {xgb_oof_f1:.4f}')

Training XGBoost 5-fold...
  Fold 1: F1=0.9424
  Fold 2: F1=0.9428
  Fold 3: F1=0.9038
  Fold 4: F1=0.9216
  Fold 5: F1=0.8739

XGBoost OOF F1: 0.9180


In [6]:
# 5-fold LightGBM OOF
print('Training LightGBM 5-fold...')

lgb_oof_probs = np.zeros((len(y_all), NUM_LABELS))
lgb_test_probs = np.zeros((len(test_df), NUM_LABELS))

for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_train_full, y_all)):
    X_tr = X_train_full[tr_idx]
    X_vl = X_train_full[vl_idx]
    y_tr = y_all[tr_idx]
    y_vl = y_all[vl_idx]
    w_tr = sample_weights_all[tr_idx]

    lgb_model = lgb.LGBMClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.3,
        min_child_samples=10,
        reg_alpha=0.1,
        reg_lambda=1.0,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    lgb_model.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_vl, y_vl)],
                  callbacks=[lgb.early_stopping(50, verbose=False)])

    lgb_oof_probs[vl_idx] = lgb_model.predict_proba(X_vl)
    lgb_test_probs += lgb_model.predict_proba(X_test_full) / 5

    fold_f1 = f1_score(y_vl, lgb_oof_probs[vl_idx].argmax(1), average='macro')
    print(f'  Fold {fold+1}: F1={fold_f1:.4f}')

lgb_oof_f1 = f1_score(y_all, lgb_oof_probs.argmax(1), average='macro')
print(f'\nLightGBM OOF F1: {lgb_oof_f1:.4f}')

Training LightGBM 5-fold...
  Fold 1: F1=0.9347
  Fold 2: F1=0.9314
  Fold 3: F1=0.8932
  Fold 4: F1=0.9270
  Fold 5: F1=0.8940

LightGBM OOF F1: 0.9165


In [7]:
# Save new OOF artifacts
np.save('models/xgb_oof_probs.npy', xgb_oof_probs)
np.save('models/xgb_test_probs.npy', xgb_test_probs)
np.save('models/lgb_oof_probs.npy', lgb_oof_probs)
np.save('models/lgb_test_probs.npy', lgb_test_probs)
print('Saved XGB and LGB OOF artifacts.')

# Summary of all base models
print(f'\n{"="*50}')
print(f'ALL BASE MODEL OOF F1 SCORES')
print(f'{"="*50}')
all_models = {
    'DeBERTa (5-fold)': (deberta_oof_probs, deberta_test_probs),
    'SVC (5-fold)': (svc_oof_probs, svc_test_probs),
    'LR (5-fold)': (lr_oof_probs, lr_test_probs),
    'XGBoost (5-fold)': (xgb_oof_probs, xgb_test_probs),
    'LightGBM (5-fold)': (lgb_oof_probs, lgb_test_probs),
}
for name, (oof, _) in all_models.items():
    f1 = f1_score(y_all, oof.argmax(1), average='macro')
    print(f'  {name:25s}: {f1:.4f}')

Saved XGB and LGB OOF artifacts.

ALL BASE MODEL OOF F1 SCORES
  DeBERTa (5-fold)         : 0.9428
  SVC (5-fold)             : 0.8979
  LR (5-fold)              : 0.9026
  XGBoost (5-fold)         : 0.9180
  LightGBM (5-fold)        : 0.9165


## 3. Exhaustive Weight Search (5 models)

With 5 models now, we search over a finer weight grid to find the optimal blend.

In [8]:
# Fine-grained weight search over 5 models
# DeBERTa is by far the strongest, so it should get the most weight
# We search: DeBERTa [0.4-0.8], SVC [0-0.3], LR [0-0.2], XGB [0-0.2], LGB [0-0.2]

print('Searching optimal weights for 5-model ensemble...')
t0 = time.time()

best_f1 = 0
best_w = None
n_combos = 0

step = 0.05
for w_d in np.arange(0.40, 0.85, step):
    for w_s in np.arange(0.0, 0.35, step):
        for w_l in np.arange(0.0, 0.25, step):
            for w_x in np.arange(0.0, 0.25, step):
                w_g = round(1.0 - w_d - w_s - w_l - w_x, 2)
                if w_g < 0 or w_g > 0.25:
                    continue
                n_combos += 1
                blend = (w_d * deberta_oof_probs +
                         w_s * svc_oof_probs +
                         w_l * lr_oof_probs +
                         w_x * xgb_oof_probs +
                         w_g * lgb_oof_probs)
                f1 = f1_score(y_all, blend.argmax(1), average='macro')
                if f1 > best_f1:
                    best_f1 = f1
                    best_w = (w_d, w_s, w_l, w_x, w_g)

elapsed = time.time() - t0
print(f'Searched {n_combos} combinations in {elapsed:.1f}s')
print(f'\n Best 5-model weighted vote:')
print(f'  DeBERTa={best_w[0]:.2f}, SVC={best_w[1]:.2f}, LR={best_w[2]:.2f}, XGB={best_w[3]:.2f}, LGB={best_w[4]:.2f}')
print(f'  OOF F1={best_f1:.4f}')
print(f'  Previous best (3-model): 0.9500')

Searching optimal weights for 5-model ensemble...
Searched 816 combinations in 0.5s

 Best 5-model weighted vote:
  DeBERTa=0.55, SVC=0.00, LR=0.00, XGB=0.20, LGB=0.25
  OOF F1=0.9557
  Previous best (3-model): 0.9500


In [9]:
# Generate the best weighted vote test predictions
w_d, w_s, w_l, w_x, w_g = best_w
weighted5_test_probs = (w_d * deberta_test_probs +
                        w_s * svc_test_probs +
                        w_l * lr_test_probs +
                        w_x * xgb_test_probs +
                        w_g * lgb_test_probs)
weighted5_test_preds = weighted5_test_probs.argmax(1)

print(f'5-model weighted vote test distribution:')
print(dict(zip(*np.unique(weighted5_test_preds, return_counts=True))))

5-model weighted vote test distribution:
{np.int64(0): np.int64(379), np.int64(1): np.int64(23), np.int64(2): np.int64(37), np.int64(3): np.int64(20), np.int64(4): np.int64(60), np.int64(5): np.int64(81)}


## 4. XGBoost Stacking Meta-Learner (5 base models)

Instead of Logistic Regression as the meta-learner, use XGBoost which can learn non-linear combinations of model probabilities.

In [10]:
# Build enhanced meta-features from 5 models
def build_meta_5models(d_probs, s_probs, l_probs, x_probs, g_probs):
    """Build meta-features from 5 base models."""
    features = []

    # Raw probabilities: 5 x 6 = 30
    features.append(d_probs)
    features.append(s_probs)
    features.append(l_probs)
    features.append(x_probs)
    features.append(g_probs)

    # Per-model confidence (5)
    for probs in [d_probs, s_probs, l_probs, x_probs, g_probs]:
        features.append(probs.max(axis=1, keepdims=True))

    # Per-model entropy (5)
    for probs in [d_probs, s_probs, l_probs, x_probs, g_probs]:
        ent = -np.sum(probs * np.log(np.clip(probs, 1e-10, 1.0)), axis=1, keepdims=True)
        features.append(ent)

    # Mean probability (6)
    mean_probs = (d_probs + s_probs + l_probs + x_probs + g_probs) / 5.0
    features.append(mean_probs)

    # Std of probabilities across models per class (6)
    stacked = np.stack([d_probs, s_probs, l_probs, x_probs, g_probs], axis=0)
    std_probs = stacked.std(axis=0)
    features.append(std_probs)

    return np.hstack(features)


meta_X_oof = build_meta_5models(deberta_oof_probs, svc_oof_probs, lr_oof_probs, xgb_oof_probs, lgb_oof_probs)
meta_X_test = build_meta_5models(deberta_test_probs, svc_test_probs, lr_test_probs, xgb_test_probs, lgb_test_probs)

print(f'Meta features: {meta_X_oof.shape[1]} (30 probs + 5 conf + 5 ent + 6 mean + 6 std)')
print(f'OOF: {meta_X_oof.shape}, Test: {meta_X_test.shape}')

Meta features: 52 (30 probs + 5 conf + 5 ent + 6 mean + 6 std)
OOF: (2400, 52), Test: (600, 52)


In [11]:
# Try multiple meta-learner types
print('Training meta-learners on 5-model features...')

meta_results = []

# Meta-learner configs
meta_configs = [
    ('LR_C0.5', LogisticRegression(C=0.5, class_weight='balanced', max_iter=5000, solver='lbfgs', random_state=SEED)),
    ('LR_C1.0', LogisticRegression(C=1.0, class_weight='balanced', max_iter=5000, solver='lbfgs', random_state=SEED)),
    ('LR_C2.0', LogisticRegression(C=2.0, class_weight='balanced', max_iter=5000, solver='lbfgs', random_state=SEED)),
    ('XGB_meta', xgb.XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        random_state=SEED, eval_metric='mlogloss',
        tree_method='hist', n_jobs=-1)),
    ('LGB_meta', lgb.LGBMClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=5,
        class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1)),
]

best_meta_name = None
best_meta_oof_f1 = 0
best_meta_test_probs = None
best_meta_oof_preds = None

for meta_name, meta_template in meta_configs:
    skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(y_all), dtype=int)
    oof_probs = np.zeros((len(y_all), NUM_LABELS))
    test_probs_sum = np.zeros((len(test_df), NUM_LABELS))
    fold_f1s = []

    for fold, (tr_idx, vl_idx) in enumerate(skf_meta.split(meta_X_oof, y_all)):
        meta_model = clone(meta_template)

        if 'XGB' in meta_name:
            w_tr = np.array([class_weight_map[y] for y in y_all[tr_idx]])
            meta_model.fit(meta_X_oof[tr_idx], y_all[tr_idx], sample_weight=w_tr)
        else:
            meta_model.fit(meta_X_oof[tr_idx], y_all[tr_idx])

        fold_probs = meta_model.predict_proba(meta_X_oof[vl_idx])
        oof_probs[vl_idx] = fold_probs
        oof_preds[vl_idx] = fold_probs.argmax(axis=1)
        fold_f1s.append(f1_score(y_all[vl_idx], oof_preds[vl_idx], average='macro'))

        test_probs_sum += meta_model.predict_proba(meta_X_test) / 5

    oof_f1 = f1_score(y_all, oof_preds, average='macro')
    print(f'{meta_name:12s}: OOF F1={oof_f1:.4f} (folds: {[f"{s:.4f}" for s in fold_f1s]})')

    meta_results.append((meta_name, oof_f1, test_probs_sum, oof_preds))

    if oof_f1 > best_meta_oof_f1:
        best_meta_oof_f1 = oof_f1
        best_meta_name = meta_name
        best_meta_test_probs = test_probs_sum.copy()
        best_meta_oof_preds = oof_preds.copy()

print(f'\n Best meta-learner: {best_meta_name}, OOF F1={best_meta_oof_f1:.4f}')

Training meta-learners on 5-model features...
LR_C0.5     : OOF F1=0.9459 (folds: ['0.9450', '0.9877', '0.9556', '0.9266', '0.9157'])
LR_C1.0     : OOF F1=0.9495 (folds: ['0.9450', '0.9877', '0.9556', '0.9200', '0.9412'])
LR_C2.0     : OOF F1=0.9476 (folds: ['0.9359', '0.9796', '0.9556', '0.9200', '0.9478'])
XGB_meta    : OOF F1=0.9454 (folds: ['0.9521', '0.9552', '0.9487', '0.9259', '0.9449'])
LGB_meta    : OOF F1=0.9430 (folds: ['0.9359', '0.9612', '0.9475', '0.9194', '0.9517'])

 Best meta-learner: LR_C1.0, OOF F1=0.9495


## 5. Per-Class Threshold Optimization

The confusion matrix showed DeepSeek↔Grok is the main confusion.
We can boost minority class recall by adjusting per-class thresholds on the probability outputs.

In [12]:
# Per-class threshold optimization on the best weighted vote
def optimize_thresholds(oof_probs, y_true, n_steps=50):
    """Find per-class probability multipliers that maximize macro F1."""
    n_classes = oof_probs.shape[1]
    best_thresholds = np.ones(n_classes)
    best_f1 = f1_score(y_true, oof_probs.argmax(1), average='macro')

    # Iteratively optimize each class threshold
    for iteration in range(3):  # multiple passes
        for cls in range(n_classes):
            best_t = best_thresholds[cls]
            for t in np.linspace(0.5, 2.0, n_steps):
                trial = best_thresholds.copy()
                trial[cls] = t
                adjusted = oof_probs * trial
                f1 = f1_score(y_true, adjusted.argmax(1), average='macro')
                if f1 > best_f1:
                    best_f1 = f1
                    best_t = t
            best_thresholds[cls] = best_t

    return best_thresholds, best_f1


# Optimize on the 5-model weighted vote OOF probs
weighted5_oof_probs = (w_d * deberta_oof_probs +
                       w_s * svc_oof_probs +
                       w_l * lr_oof_probs +
                       w_x * xgb_oof_probs +
                       w_g * lgb_oof_probs)

thresholds, thresh_f1 = optimize_thresholds(weighted5_oof_probs, y_all)
print(f'Per-class thresholds:')
for cls in range(NUM_LABELS):
    print(f'  {LABEL_MAP[cls]:10s}: {thresholds[cls]:.3f}')
print(f'\nBefore thresholds: OOF F1={best_f1:.4f}')
print(f'After thresholds:  OOF F1={thresh_f1:.4f}')

# Apply to test
thresh_test_preds = (weighted5_test_probs * thresholds).argmax(1)
print(f'\nThresholded test distribution: {dict(zip(*np.unique(thresh_test_preds, return_counts=True)))}')

Per-class thresholds:
  Human     : 1.204
  DeepSeek  : 1.265
  Grok      : 1.000
  Claude    : 1.786
  Gemini    : 1.000
  ChatGPT   : 1.000

Before thresholds: OOF F1=0.9557
After thresholds:  OOF F1=0.9591

Thresholded test distribution: {np.int64(0): np.int64(379), np.int64(1): np.int64(26), np.int64(2): np.int64(34), np.int64(3): np.int64(20), np.int64(4): np.int64(60), np.int64(5): np.int64(81)}


In [13]:
# Also optimize thresholds on the stacking output
if best_meta_test_probs is not None:
    # Reconstruct stacking OOF probs
    # We need to redo stacking to get OOF probs (not just preds)
    # Use the best meta result
    for name, f1, test_p, oof_p in meta_results:
        if name == best_meta_name:
            stack_oof_preds = oof_p
            break

    # Build stacking OOF probs from meta CV
    skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    stack_oof_probs = np.zeros((len(y_all), NUM_LABELS))
    for meta_name_i, meta_template in meta_configs:
        if meta_name_i != best_meta_name:
            continue
        for fold, (tr_idx, vl_idx) in enumerate(skf_meta.split(meta_X_oof, y_all)):
            meta_model = clone(meta_template)
            if 'XGB' in meta_name_i:
                w_tr = np.array([class_weight_map[y] for y in y_all[tr_idx]])
                meta_model.fit(meta_X_oof[tr_idx], y_all[tr_idx], sample_weight=w_tr)
            else:
                meta_model.fit(meta_X_oof[tr_idx], y_all[tr_idx])
            stack_oof_probs[vl_idx] = meta_model.predict_proba(meta_X_oof[vl_idx])

    stack_thresh, stack_thresh_f1 = optimize_thresholds(stack_oof_probs, y_all)
    print(f'Stacking + threshold optimization:')
    print(f'  Before: OOF F1={best_meta_oof_f1:.4f}')
    print(f'  After:  OOF F1={stack_thresh_f1:.4f}')
    for cls in range(NUM_LABELS):
        print(f'    {LABEL_MAP[cls]:10s}: {stack_thresh[cls]:.3f}')

    stack_thresh_test_preds = (best_meta_test_probs * stack_thresh).argmax(1)

Stacking + threshold optimization:
  Before: OOF F1=0.9495
  After:  OOF F1=0.9495
    Human     : 1.000
    DeepSeek  : 1.000
    Grok      : 1.000
    Claude    : 1.000
    Gemini    : 1.000
    ChatGPT   : 1.000


## 6. Grand Comparison & Submit

Compare all V3 methods and submit the best ones.

In [14]:
# Collect all methods
print('=' * 65)
print('V3 GRAND COMPARISON')
print('=' * 65)

methods = []

# V2 baseline
v2_blend = 0.65 * deberta_oof_probs + 0.25 * svc_oof_probs + 0.10 * lr_oof_probs
v2_f1 = f1_score(y_all, v2_blend.argmax(1), average='macro')
v2_test = 0.65 * deberta_test_probs + 0.25 * svc_test_probs + 0.10 * lr_test_probs
methods.append(('V2 weighted (3-model)', v2_f1, v2_test.argmax(1)))

# 5-model weighted vote
methods.append((f'V3 weighted (5-model)', best_f1, weighted5_test_preds))

# 5-model weighted + thresholds
methods.append(('V3 weighted + thresh', thresh_f1, thresh_test_preds))

# Stacking
methods.append((f'V3 stacking ({best_meta_name})', best_meta_oof_f1, best_meta_test_probs.argmax(1)))

# Stacking + thresholds
if 'stack_thresh_f1' in dir():
    methods.append(('V3 stacking + thresh', stack_thresh_f1, stack_thresh_test_preds))

# DeBERTa only
d_f1 = f1_score(y_all, deberta_oof_probs.argmax(1), average='macro')
methods.append(('DeBERTa only', d_f1, deberta_test_probs.argmax(1)))

# Sort by OOF F1
methods.sort(key=lambda x: x[1], reverse=True)

print(f'{"Method":40s} {"OOF F1":>8}')
print('-' * 50)
for name, f1, _ in methods:
    marker = ' ' if f1 == methods[0][1] else ''
    print(f'{name:40s} {f1:.4f}{marker}')

print(f'\nPrevious LB best: 0.92170')
print(f'Target (1st place): 0.95081')

V3 GRAND COMPARISON
Method                                     OOF F1
--------------------------------------------------
V3 weighted + thresh                     0.9591 
V3 weighted (5-model)                    0.9557
V2 weighted (3-model)                    0.9500
V3 stacking (LR_C1.0)                    0.9495
V3 stacking + thresh                     0.9495
DeBERTa only                             0.9428

Previous LB best: 0.92170
Target (1st place): 0.95081


In [15]:
# Write submissions for the top methods
os.makedirs('submissions', exist_ok=True)

def write_sub(preds, path, name):
    lines = ['ID,LABEL'] + [f'{i},{int(preds[i])}' for i in range(600)]
    with open(path, 'w', newline='\n') as f:
        f.write('\n'.join(lines) + '\n')
    dist = dict(zip(*np.unique(preds, return_counts=True)))
    print(f'  {name}: {path} | dist={dist}')


print('Writing V3 submissions:\n')

# Submit top 3 methods
for i, (name, f1, preds) in enumerate(methods[:3]):
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')
    path = f'submissions/v3_{safe_name}.csv'
    write_sub(preds, path, f'{name} (F1={f1:.4f})')

# Also compare test predictions
print(f'\nAgreement between top 2:')
if len(methods) >= 2:
    a = methods[0][2]
    b = methods[1][2]
    agree = (a == b).mean() * 100
    print(f'  {methods[0][0]} vs {methods[1][0]}: {agree:.1f}%')

Writing V3 submissions:

  V3 weighted + thresh (F1=0.9591): submissions/v3_v3_weighted_+_thresh.csv | dist={np.int64(0): np.int64(379), np.int64(1): np.int64(26), np.int64(2): np.int64(34), np.int64(3): np.int64(20), np.int64(4): np.int64(60), np.int64(5): np.int64(81)}
  V3 weighted (5-model) (F1=0.9557): submissions/v3_v3_weighted_5model.csv | dist={np.int64(0): np.int64(379), np.int64(1): np.int64(23), np.int64(2): np.int64(37), np.int64(3): np.int64(20), np.int64(4): np.int64(60), np.int64(5): np.int64(81)}
  V2 weighted (3-model) (F1=0.9500): submissions/v3_v2_weighted_3model.csv | dist={np.int64(0): np.int64(379), np.int64(1): np.int64(25), np.int64(2): np.int64(35), np.int64(3): np.int64(20), np.int64(4): np.int64(60), np.int64(5): np.int64(81)}

Agreement between top 2:
  V3 weighted + thresh vs V3 weighted (5-model): 99.5%


In [16]:
# Submit to Kaggle from notebook (using system kaggle CLI)
import subprocess

COMPETITION = 'malto-recruitment-hackathon'
env = os.environ.copy()
env['KAGGLE_API_TOKEN'] = 'KGAT_b4a6c46509414f03c35239d14c1e1770'
env['PATH'] = env.get('PATH', '') + ':/Users/aliivaezii/Library/Python/3.9/bin'

kaggle_bin = '/Users/aliivaezii/Library/Python/3.9/bin/kaggle'

for i, (name, f1, preds) in enumerate(methods[:3]):
    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')
    path = f'submissions/v3_{safe_name}.csv'
    msg = f'V3: {name}, OOF F1={f1:.4f}'

    print(f'Submitting: {name}...')
    result = subprocess.run(
        [kaggle_bin, 'competitions', 'submit', '-c', COMPETITION, '-f', path, '-m', msg],
        capture_output=True, text=True, env=env
    )
    if result.returncode == 0:
        print(f'   Success')
    else:
        print(f'   Error: {result.stderr.strip()[-100:]}')

print('\nWaiting 30s for scores...')
time.sleep(30)

# Check scores
result = subprocess.run(
    [kaggle_bin, 'competitions', 'submissions', '-c', COMPETITION],
    capture_output=True, text=True, env=env
)
print(result.stdout[:2000])

Submitting: V3 weighted + thresh...
   Error: Error('Error: Missing %s in configuration.' % item)
ValueError: Error: Missing key in configuration.
Submitting: V3 weighted (5-model)...
   Error: Error('Error: Missing %s in configuration.' % item)
ValueError: Error: Missing key in configuration.
Submitting: V2 weighted (3-model)...
   Error: Error('Error: Missing %s in configuration.' % item)
ValueError: Error: Missing key in configuration.

Waiting 30s for scores...



In [17]:
# Save V3 config
v3_config = {
    'version': 'v3',
    'improvements': [
        'Added XGBoost 5-fold OOF',
        'Added LightGBM 5-fold OOF',
        '5-model weighted vote search',
        '5-model stacking with XGB/LGB meta-learners',
        'Per-class threshold optimization',
    ],
    'base_models': {
        'deberta': f1_score(y_all, deberta_oof_probs.argmax(1), average='macro'),
        'svc': f1_score(y_all, svc_oof_probs.argmax(1), average='macro'),
        'lr': f1_score(y_all, lr_oof_probs.argmax(1), average='macro'),
        'xgboost': xgb_oof_f1,
        'lightgbm': lgb_oof_f1,
    },
    'best_weighted': {
        'weights': dict(zip(['deberta', 'svc', 'lr', 'xgb', 'lgb'], best_w)),
        'oof_f1': best_f1,
    },
    'best_stacking': {
        'meta_learner': best_meta_name,
        'oof_f1': best_meta_oof_f1,
    },
    'ranked_methods': [(name, float(f1)) for name, f1, _ in methods],
}

with open('models/v3_config.json', 'w') as f:
    json.dump(v3_config, f, indent=2)

print('Saved: models/v3_config.json')
print(f'\nBest V3 method: {methods[0][0]} (OOF F1={methods[0][1]:.4f})')

Saved: models/v3_config.json

Best V3 method: V3 weighted + thresh (OOF F1=0.9591)
